In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
%sql
USE CATALOG skylens_macro;
USE SCHEMA team1;

In [0]:
CHECKPOINTS_PATH = "/Volumes/Skylens_macro/team1/checkpoints/gold"

In [0]:
df_silver_flights = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")  
    .table("workspace.default.silver_flights")
)

df_silver_airlines = spark.read.table("workspace.default.silver_airlines")   # small dim table — batch is fine
df_silver_airports = spark.read.table("workspace.default.silver_airports")   # small dim table — batch is fine

In [0]:
def write_gold_airline_performance(batch_df, batch_id):

    agg = batch_df.groupBy("AIRLINE", "airline_full_name").agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("CANCELLED") == 1, 1).otherwise(0)).alias("total_cancelled"),
        F.sum(F.when(F.col("DIVERTED") == 1, 1).otherwise(0)).alias("total_diverted"),
        F.round(F.avg(F.when(F.col("CANCELLED") == 0, F.col("ARRIVAL_DELAY"))), 2).alias("avg_arrival_delay_min"),
        F.round(
            F.sum(F.when((F.col("CANCELLED") == 0) & (F.col("ARRIVAL_DELAY") <= 15), 1).otherwise(0)) * 100.0
            / F.count("*"), 2
        ).alias("on_time_pct"),
        F.round(F.avg(F.col("taxi_total_minutes")), 2).alias("avg_taxi_minutes"),
        F.round(F.avg(F.col("total_delay_minutes")), 2).alias("avg_total_delay_min"),
        F.round(F.sum(F.col("WEATHER_DELAY")), 2).alias("total_weather_delay_min"),
        F.round(F.sum(F.col("AIRLINE_DELAY")), 2).alias("total_airline_delay_min"),
        F.round(F.sum(F.col("AIR_SYSTEM_DELAY")), 2).alias("total_air_system_delay_min"),
        F.round(F.sum(F.col("LATE_AIRCRAFT_DELAY")), 2).alias("total_late_aircraft_delay_min"),
        F.max(F.col("ingestion_timestamp")).alias("last_updated")
    )

    # Create table first time if it doesn't exist
    if not spark.catalog.tableExists("Skylens_macro.team1.gold_airline_performance"):
        agg.write.format("delta").option("mergeSchema", "true") \
            .saveAsTable("Skylens_macro.team1.gold_airline_performance")
        return

    gold_table = DeltaTable.forName(spark, "Skylens_macro.team1.gold_airline_performance")

    (
        gold_table.alias("gold")
        .merge(
            agg.alias("new"),
            "gold.AIRLINE = new.AIRLINE"                      # merge key = airline code
        )
        .whenMatchedUpdate(set={
            "total_flights"               : "gold.total_flights + new.total_flights",
            "total_cancelled"             : "gold.total_cancelled + new.total_cancelled",
            "total_diverted"              : "gold.total_diverted + new.total_diverted",
            "total_weather_delay_min"     : "gold.total_weather_delay_min + new.total_weather_delay_min",
            "total_airline_delay_min"     : "gold.total_airline_delay_min + new.total_airline_delay_min",
            "total_air_system_delay_min"  : "gold.total_air_system_delay_min + new.total_air_system_delay_min",
            "total_late_aircraft_delay_min": "gold.total_late_aircraft_delay_min + new.total_late_aircraft_delay_min",
            # avg and % — recalculate properly using running totals
            "avg_arrival_delay_min"       : "new.avg_arrival_delay_min",
            "on_time_pct"                 : "new.on_time_pct",
            "avg_taxi_minutes"            : "new.avg_taxi_minutes",
            "avg_total_delay_min"         : "new.avg_total_delay_min",
            "last_updated"                : "new.last_updated"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df_silver_flights
    .writeStream
    .foreachBatch(write_gold_airline_performance)
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/airline_performance/")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
def write_gold_route_performance(batch_df, batch_id):

    agg = batch_df.groupBy(
        "route_id",
        "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
        "origin_city", "origin_state",
        "dest_city", "dest_state"
    ).agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("CANCELLED") == 1, 1).otherwise(0)).alias("total_cancelled"),
        F.round(F.avg(F.when(F.col("CANCELLED") == 0, F.col("ARRIVAL_DELAY"))), 2).alias("avg_arrival_delay_min"),
        F.round(
            F.sum(F.when((F.col("CANCELLED") == 0) & (F.col("ARRIVAL_DELAY") <= 15), 1).otherwise(0)) * 100.0
            / F.count("*"), 2
        ).alias("on_time_pct"),
        F.round(F.avg(F.col("AIR_TIME")), 2).alias("avg_air_time_min"),
        F.round(F.avg(F.col("DISTANCE")), 2).alias("avg_distance_miles"),
        F.max(F.col("ingestion_timestamp")).alias("last_updated")
    )

    if not spark.catalog.tableExists("Skylens_macro.team1.gold_route_performance"):
        agg.write.format("delta").option("mergeSchema", "true") \
            .saveAsTable("Skylens_macro.team1.gold_route_performance")
        return

    gold_table = DeltaTable.forName(spark, "Skylens_macro.team1.gold_route_performance")

    (
        gold_table.alias("gold")
        .merge(
            agg.alias("new"),
            "gold.route_id = new.route_id"                    # merge key = route
        )
        .whenMatchedUpdate(set={
            "total_flights"         : "gold.total_flights + new.total_flights",
            "total_cancelled"       : "gold.total_cancelled + new.total_cancelled",
            "avg_arrival_delay_min" : "new.avg_arrival_delay_min",
            "on_time_pct"           : "new.on_time_pct",
            "avg_air_time_min"      : "new.avg_air_time_min",
            "avg_distance_miles"    : "new.avg_distance_miles",
            "last_updated"          : "new.last_updated"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df_silver_flights
    .writeStream
    .foreachBatch(write_gold_route_performance)
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/route_performance/")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
def write_gold_airport_traffic(batch_df, batch_id):

    departures = batch_df.groupBy(
        F.col("ORIGIN_AIRPORT").alias("airport_code"),
        F.col("origin_city").alias("city"),
        F.col("origin_state").alias("state")
    ).agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("CANCELLED") == 1, 1).otherwise(0)).alias("total_cancelled"),
        F.round(F.avg(F.when(F.col("CANCELLED") == 0, F.col("DEPARTURE_DELAY"))), 2).alias("avg_delay_min"),
        F.max(F.col("ingestion_timestamp")).alias("last_updated")
    ).withColumn("traffic_type", F.lit("DEPARTURE"))

    arrivals = batch_df.groupBy(
        F.col("DESTINATION_AIRPORT").alias("airport_code"),
        F.col("dest_city").alias("city"),
        F.col("dest_state").alias("state")
    ).agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("CANCELLED") == 1, 1).otherwise(0)).alias("total_cancelled"),
        F.round(F.avg(F.when(F.col("CANCELLED") == 0, F.col("ARRIVAL_DELAY"))), 2).alias("avg_delay_min"),
        F.max(F.col("ingestion_timestamp")).alias("last_updated")
    ).withColumn("traffic_type", F.lit("ARRIVAL"))

    combined = departures.union(arrivals)

    if not spark.catalog.tableExists("Skylens_macro.team1.gold_airport_traffic"):
        combined.write.format("delta").option("mergeSchema", "true") \
            .saveAsTable("Skylens_macro.team1.gold_airport_traffic")
        return

    gold_table = DeltaTable.forName(spark, "Skylens_macro.team1.gold_airport_traffic")

    (
        gold_table.alias("gold")
        .merge(
            combined.alias("new"),
            "gold.airport_code = new.airport_code AND gold.traffic_type = new.traffic_type"
        )
        .whenMatchedUpdate(set={
            "total_flights"   : "gold.total_flights + new.total_flights",
            "total_cancelled" : "gold.total_cancelled + new.total_cancelled",
            "avg_delay_min"   : "new.avg_delay_min",
            "last_updated"    : "new.last_updated"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df_silver_flights
    .writeStream
    .foreachBatch(write_gold_airport_traffic)
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/airport_traffic/")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
def write_gold_monthly_trends(batch_df, batch_id):

    agg = batch_df \
        .withColumn("flight_month", F.date_format(F.col("flight_date"), "yyyy-MM")) \
        .groupBy("flight_month", "MONTH", "flight_quarter") \
        .agg(
            F.count("*").alias("total_flights"),
            F.sum(F.when(F.col("CANCELLED") == 1, 1).otherwise(0)).alias("total_cancelled"),
            F.sum(F.when(F.col("DIVERTED") == 1, 1).otherwise(0)).alias("total_diverted"),
            F.round(F.avg(F.when(F.col("CANCELLED") == 0, F.col("ARRIVAL_DELAY"))), 2).alias("avg_arrival_delay_min"),
            F.round(F.avg(F.col("total_delay_minutes")), 2).alias("avg_total_delay_min"),
            F.round(
                F.sum(F.when((F.col("CANCELLED") == 0) & (F.col("ARRIVAL_DELAY") <= 15), 1).otherwise(0)) * 100.0
                / F.count("*"), 2
            ).alias("on_time_pct"),
            F.round(F.sum(F.col("WEATHER_DELAY")), 2).alias("total_weather_delay_min"),
            F.round(F.sum(F.col("AIRLINE_DELAY")), 2).alias("total_airline_delay_min"),
            F.max(F.col("ingestion_timestamp")).alias("last_updated")
        )

    if not spark.catalog.tableExists("Skylens_macro.team1.gold_monthly_trends"):
        agg.write.format("delta").option("mergeSchema", "true") \
            .partitionBy("flight_month") \
            .saveAsTable("Skylens_macro.team1.gold_monthly_trends")
        return

    gold_table = DeltaTable.forName(spark, "Skylens_macro.team1.gold_monthly_trends")

    (
        gold_table.alias("gold")
        .merge(
            agg.alias("new"),
            "gold.flight_month = new.flight_month"            # merge key = month
        )
        .whenMatchedUpdate(set={
            "total_flights"           : "gold.total_flights + new.total_flights",
            "total_cancelled"         : "gold.total_cancelled + new.total_cancelled",
            "total_diverted"          : "gold.total_diverted + new.total_diverted",
            "total_weather_delay_min" : "gold.total_weather_delay_min + new.total_weather_delay_min",
            "total_airline_delay_min" : "gold.total_airline_delay_min + new.total_airline_delay_min",
            "avg_arrival_delay_min"   : "new.avg_arrival_delay_min",
            "avg_total_delay_min"     : "new.avg_total_delay_min",
            "on_time_pct"             : "new.on_time_pct",
            "last_updated"            : "new.last_updated"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df_silver_flights
    .writeStream
    .foreachBatch(write_gold_monthly_trends)
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/monthly_trends/")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
def write_gold_delay_cause_breakdown(batch_df, batch_id):

    agg = batch_df \
        .filter(F.col("CANCELLED") == 0) \
        .groupBy("AIRLINE", "airline_full_name") \
        .agg(
            F.count("*").alias("total_operated_flights"),
            F.round(F.sum(F.col("WEATHER_DELAY")), 2).alias("total_weather_delay_min"),
            F.round(F.sum(F.col("AIRLINE_DELAY")), 2).alias("total_airline_delay_min"),
            F.round(F.sum(F.col("AIR_SYSTEM_DELAY")), 2).alias("total_air_system_delay_min"),
            F.round(F.sum(F.col("SECURITY_DELAY")), 2).alias("total_security_delay_min"),
            F.round(F.sum(F.col("LATE_AIRCRAFT_DELAY")), 2).alias("total_late_aircraft_delay_min"),
            F.round(F.sum(F.col("total_delay_minutes")), 2).alias("grand_total_delay_min"),
            F.round(F.avg(F.col("total_delay_minutes")), 2).alias("avg_delay_per_flight_min"),
            F.max(F.col("ingestion_timestamp")).alias("last_updated")
        )

    if not spark.catalog.tableExists("Skylens_macro.team1.gold_delay_cause_breakdown"):
        agg.write.format("delta").option("mergeSchema", "true") \
            .saveAsTable("Skylens_macro.team1.gold_delay_cause_breakdown")
        return

    gold_table = DeltaTable.forName(spark, "Skylens_macro.team1.gold_delay_cause_breakdown")

    (
        gold_table.alias("gold")
        .merge(
            agg.alias("new"),
            "gold.AIRLINE = new.AIRLINE"
        )
        .whenMatchedUpdate(set={
            "total_operated_flights"        : "gold.total_operated_flights + new.total_operated_flights",
            "total_weather_delay_min"       : "gold.total_weather_delay_min + new.total_weather_delay_min",
            "total_airline_delay_min"       : "gold.total_airline_delay_min + new.total_airline_delay_min",
            "total_air_system_delay_min"    : "gold.total_air_system_delay_min + new.total_air_system_delay_min",
            "total_security_delay_min"      : "gold.total_security_delay_min + new.total_security_delay_min",
            "total_late_aircraft_delay_min" : "gold.total_late_aircraft_delay_min + new.total_late_aircraft_delay_min",
            "grand_total_delay_min"         : "gold.grand_total_delay_min + new.grand_total_delay_min",
            "avg_delay_per_flight_min"      : "new.avg_delay_per_flight_min",
            "last_updated"                  : "new.last_updated"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df_silver_flights
    .writeStream
    .foreachBatch(write_gold_delay_cause_breakdown)
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/delay_cause_breakdown/")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
def write_gold_cancellation_summary(batch_df, batch_id):

    total_per_airline = batch_df.groupBy("AIRLINE").agg(
        F.count("*").alias("total_flights")
    )

    agg = batch_df \
        .filter(F.col("CANCELLED") == 1) \
        .groupBy("AIRLINE", "airline_full_name", "cancellation_reason_mapped") \
        .agg(
            F.count("*").alias("total_cancellations"),
            F.max(F.col("ingestion_timestamp")).alias("last_updated")
        ) \
        .join(total_per_airline, on="AIRLINE", how="left") \
        .withColumn(
            "cancellation_rate_pct",
            F.round(F.col("total_cancellations") * 100.0 / F.col("total_flights"), 2)
        )

    if not spark.catalog.tableExists("Skylens_macro.team1.gold_cancellation_summary"):
        agg.write.format("delta").option("mergeSchema", "true") \
            .saveAsTable("Skylens_macro.team1.gold_cancellation_summary")
        return

    gold_table = DeltaTable.forName(spark, "Skylens_macro.team1.gold_cancellation_summary")

    (
        gold_table.alias("gold")
        .merge(
            agg.alias("new"),
            # merge key = airline + reason combination
            "gold.AIRLINE = new.AIRLINE AND gold.cancellation_reason_mapped = new.cancellation_reason_mapped"
        )
        .whenMatchedUpdate(set={
            "total_cancellations"   : "gold.total_cancellations + new.total_cancellations",
            "total_flights"         : "gold.total_flights + new.total_flights",
            "cancellation_rate_pct" : "new.cancellation_rate_pct",
            "last_updated"          : "new.last_updated"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df_silver_flights
    .writeStream
    .foreachBatch(write_gold_cancellation_summary)
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/cancellation_summary/")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
%sql
SELECT 'gold_airline_performance'    AS table_name, COUNT(*) AS row_count FROM Skylens_macro.team1.gold_airline_performance
UNION ALL
SELECT 'gold_route_performance',     COUNT(*) FROM Skylens_macro.team1.gold_route_performance
UNION ALL
SELECT 'gold_airport_traffic',       COUNT(*) FROM Skylens_macro.team1.gold_airport_traffic
UNION ALL
SELECT 'gold_monthly_trends',        COUNT(*) FROM Skylens_macro.team1.gold_monthly_trends
UNION ALL
SELECT 'gold_delay_cause_breakdown', COUNT(*) FROM Skylens_macro.team1.gold_delay_cause_breakdown
UNION ALL
SELECT 'gold_cancellation_summary',  COUNT(*) FROM Skylens_macro.team1.gold_cancellation_summary;

In [0]:
from pyspark.sql.types import DateType
import datetime


start_date = datetime.date(2010, 1, 1)
end_date   = datetime.date(2025, 12, 31)

date_range = [(start_date + datetime.timedelta(days=i),)
              for i in range((end_date - start_date).days + 1)]

df_dates = spark.createDataFrame(date_range, ["full_date"]) \
    .withColumn("full_date", F.col("full_date").cast(DateType()))

# US Federal Holidays (fixed dates only — good enough for flight analysis)
us_holidays = [
    "01-01",  # New Year's Day
    "07-04",  # Independence Day
    "11-11",  # Veterans Day
    "12-25",  # Christmas
    "12-24",  # Christmas Eve
    "11-26",  # Thanksgiving approximate — we handle this separately below
]

dim_date = df_dates \
    .withColumn("date_key",     F.date_format(F.col("full_date"), "yyyyMMdd").cast("int")) \
    .withColumn("year",         F.year("full_date")) \
    .withColumn("month",        F.month("full_date")) \
    .withColumn("month_name",   F.date_format(F.col("full_date"), "MMMM")) \
    .withColumn("quarter",      F.quarter("full_date")) \
    .withColumn("week_of_year", F.weekofyear("full_date")) \
    .withColumn("day_of_week",  F.dayofweek("full_date")) \
    .withColumn("day_name",     F.date_format(F.col("full_date"), "EEEE")) \
    .withColumn("is_weekend",   F.col("day_of_week").isin([1, 7])) \
    .withColumn("month_day",    F.date_format(F.col("full_date"), "MM-dd")) \
    .withColumn("is_us_holiday", F.col("month_day").isin(us_holidays)) \
    .drop("month_day")

dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Skylens_macro.team1.dim_date")

print("✅ dim_date done —", dim_date.count(), "rows")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

data = [
    (1, "A", "Airline/Carrier"),
    (2, "B", "Weather"),
    (3, "C", "National Air System"),
    (4, "D", "Security"),
    (5, "N", "Not Cancelled"),
]

schema = StructType([
    StructField("cancellation_key",  IntegerType(), False),
    StructField("cancellation_code", StringType(),  True),
    StructField("cancellation_desc", StringType(),  True),
])

dim_cancellation = spark.createDataFrame(data, schema)

dim_cancellation.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Skylens_macro.team1.dim_cancellation_reason")

print("✅ dim_cancellation_reason done")

In [0]:
data = [
    (1, "WEATHER",       "Weather Delay"),
    (2, "AIRLINE",       "Airline/Carrier Delay"),
    (3, "AIR_SYSTEM",    "National Air System Delay"),
    (4, "SECURITY",      "Security Delay"),
    (5, "LATE_AIRCRAFT", "Late Aircraft Delay"),
]

schema = StructType([
    StructField("delay_type_key",  IntegerType(), False),
    StructField("delay_type_code", StringType(),  True),
    StructField("delay_type_desc", StringType(),  True),
])

dim_delay_type = spark.createDataFrame(data, schema)

dim_delay_type.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Skylens_macro.team1.dim_delay_type")

print("✅ dim_delay_type done")

In [0]:
window_spec = Window.orderBy("iata_code")

dim_airline = df_silver_airlines \
    .select(
        F.col("airline_code").alias("iata_code"),
        F.col("AIRLINE").alias("airline_name")
    ) \
    .dropDuplicates(["iata_code"]) \
    .withColumn("airline_key", F.row_number().over(window_spec))

dim_airline.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Skylens_macro.team1.dim_airline")



In [0]:
window_spec = Window.orderBy("iata_code")

dim_origin_airport = df_silver_airports \
    .select(
        F.col("airport_code").alias("iata_code"),
        F.col("airport_name"),
        F.col("CITY").alias("city"),
        F.col("STATE").alias("state"),
        F.col("COUNTRY").alias("country"),
        F.col("LATITUDE").alias("latitude"),
        F.col("LONGITUDE").alias("longitude")
    ) \
    .dropDuplicates(["iata_code"]) \
    .withColumn("airport_key", F.row_number().over(window_spec))

dim_origin_airport.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Skylens_macro.team1.dim_origin_airport")

print("✅ dim_origin_airport done —", dim_origin_airport.count(), "rows")

In [0]:
dim_destination_airport = df_silver_airports \
    .select(
        F.col("airport_code").alias("iata_code"),
        F.col("airport_name"),
        F.col("CITY").alias("city"),
        F.col("STATE").alias("state"),
        F.col("COUNTRY").alias("country"),
        F.col("LATITUDE").alias("latitude"),
        F.col("LONGITUDE").alias("longitude")
    ) \
    .dropDuplicates(["iata_code"]) \
    .withColumn("airport_key", F.row_number().over(window_spec))

dim_destination_airport.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Skylens_macro.team1.dim_destination_airport")

print("✅ dim_destination_airport done —", dim_destination_airport.count(), "rows")

In [0]:
df_silver_flights=spark.read.table('workspace.default.silver_flights')

In [0]:
window_spec = Window.orderBy("tail_number")

dim_aircraft = df_silver_flights \
    .select(
        F.col("TAIL_NUMBER").alias("tail_number"),
        F.col("AIRLINE").alias("airline_iata_code")
    ) \
    .dropDuplicates(["tail_number"]) \
    .withColumn("aircraft_key", F.row_number().over(window_spec))

dim_aircraft.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Skylens_macro.team1.dim_aircraft")

print("✅ dim_aircraft done —", dim_aircraft.count(), "rows")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

unknown_row = spark.createDataFrame(
    [("UNKNOWN", "Unknown Airport", "Unknown", "Unknown", "Unknown", 0.0, 0.0, 0)],
    StructType([
        StructField("iata_code", StringType(), True),
        StructField("airport_name", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("country", StringType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("airport_key", IntegerType(), True)
    ])
)

existing_origin = spark.read.table("Skylens_macro.team1.dim_origin_airport")
existing_dest   = spark.read.table("Skylens_macro.team1.dim_destination_airport")

existing_origin.union(unknown_row).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Skylens_macro.team1.dim_origin_airport")

existing_dest.union(unknown_row).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Skylens_macro.team1.dim_destination_airport")

print("✅ UNKNOWN airport (key=0) added to both dims")

In [0]:
dim_date_df     = spark.read.table("Skylens_macro.team1.dim_date")
dim_airline_df  = spark.read.table("Skylens_macro.team1.dim_airline")
dim_origin_df   = spark.read.table("Skylens_macro.team1.dim_origin_airport")
dim_dest_df     = spark.read.table("Skylens_macro.team1.dim_destination_airport")
dim_aircraft_df = spark.read.table("Skylens_macro.team1.dim_aircraft")
dim_cancel_df   = spark.read.table("Skylens_macro.team1.dim_cancellation_reason")

fact = (df_silver_flights \

    .join(
        dim_date_df.select("date_key", "full_date"),
        df_silver_flights.flight_date == dim_date_df.full_date,
        "left"
    ) \

    .join(
        dim_airline_df.select(
            F.col("airline_key"),
            F.col("iata_code").alias("airline_iata_join")
        ),
        df_silver_flights.AIRLINE == F.col("airline_iata_join"),
        "left"
    ) \

    .join(
        dim_origin_df.select(
            F.col("airport_key").alias("origin_airport_key"),
            F.col("iata_code").alias("origin_iata_join")
        ),
        df_silver_flights.ORIGIN_AIRPORT == F.col("origin_iata_join"),
        "left"
    ) \

    .join(
        dim_dest_df.select(
            F.col("airport_key").alias("dest_airport_key"),
            F.col("iata_code").alias("dest_iata_join")
        ),
        df_silver_flights.DESTINATION_AIRPORT == F.col("dest_iata_join"),
        "left"
    ) \

    .join(
        dim_aircraft_df.select(
            F.col("aircraft_key"),
            F.col("tail_number").alias("tail_join")
        ),
        df_silver_flights.TAIL_NUMBER == F.col("tail_join"),
        "left"
    ) \

    .join(
        dim_cancel_df.select("cancellation_key", "cancellation_code"),
        df_silver_flights.CANCELLATION_REASON == F.col("cancellation_code"),
        "left"
    )
)
# handle nulls — numeric airport codes get key=0 (UNKNOWN)
# non-cancelled flights get cancellation_key=5 (Not Cancelled)
fact = fact \
    .withColumn("origin_airport_key",
        F.coalesce(F.col("origin_airport_key"), F.lit(0))
    ) \
    .withColumn("dest_airport_key",
        F.coalesce(F.col("dest_airport_key"), F.lit(0))
    ) \
    .withColumn("cancellation_key",
        F.when(F.col("CANCELLED") == 0, F.lit(5))
         .otherwise(F.coalesce(F.col("cancellation_key"), F.lit(5)))
    )

window_spec = Window.orderBy(
    "flight_date", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT"
)

fact_flights = fact \
    .withColumn("flight_key", F.row_number().over(window_spec)) \
    .select(
        F.col("flight_key"),
        F.col("date_key"),
        F.col("airline_key"),
        F.col("origin_airport_key"),
        F.col("dest_airport_key"),
        F.col("aircraft_key"),
        F.col("cancellation_key"),
        F.col("FLIGHT_NUMBER").alias("flight_number"),
        F.col("route_id"),
        F.col("scheduled_departure_ts"),
        F.col("actual_departure_ts"),
        F.col("DEPARTURE_DELAY").alias("departure_delay_min"),
        F.col("ARRIVAL_DELAY").alias("arrival_delay_min"),
        F.col("TAXI_OUT").alias("taxi_out_min"),
        F.col("TAXI_IN").alias("taxi_in_min"),
        F.col("DISTANCE").alias("distance_miles"),
        F.col("ELAPSED_TIME").alias("elapsed_time_min"),
        F.col("AIR_TIME").alias("air_time_min"),
        F.col("WEATHER_DELAY").alias("weather_delay_min"),
        F.col("AIRLINE_DELAY").alias("airline_delay_min"),
        F.col("AIR_SYSTEM_DELAY").alias("nas_delay_min"),
        F.col("SECURITY_DELAY").alias("security_delay_min"),
        F.col("LATE_AIRCRAFT_DELAY").alias("late_aircraft_delay_min"),
        F.col("CANCELLED").cast("boolean").alias("is_cancelled"),
        F.col("DIVERTED").cast("boolean").alias("is_diverted"),
        F.col("delay_category"),
        F.col("source_file_name"),
        F.col("batch_id")
    )

fact_flights.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("date_key") \
    .saveAsTable("Skylens_macro.team1.fact_flights")

print("✅ fact_flights rebuilt —", fact_flights.count(), "rows")

In [0]:
fact_flights_df = spark.read.table("Skylens_macro.team1.fact_flights")

# join back to get origin and dest airport keys cleanly
fact_daily = fact_flights_df \
    .groupBy(
        "date_key",
        "origin_airport_key",
        "dest_airport_key",
        "route_id"
    ).agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(F.col("is_cancelled") == True, 1).otherwise(0)).alias("cancelled_flights"),
        F.sum(F.when(F.col("is_diverted") == True, 1).otherwise(0)).alias("diverted_flights"),
        F.sum(F.when(F.col("arrival_delay_min") <= 0, 1).otherwise(0)).alias("on_time_flights"),
        F.round(F.avg(F.when(F.col("is_cancelled") == False, F.col("departure_delay_min"))), 2).alias("avg_departure_delay_min"),
        F.round(F.avg(F.when(F.col("is_cancelled") == False, F.col("arrival_delay_min"))), 2).alias("avg_arrival_delay_min"),
        F.max(F.col("arrival_delay_min")).alias("max_arrival_delay_min"),
        F.round(F.avg(F.col("distance_miles")), 2).alias("avg_distance_miles"),
        F.round(F.avg(
            F.coalesce(F.col("taxi_out_min"), F.lit(0)) +
            F.coalesce(F.col("taxi_in_min"), F.lit(0))
        ), 2).alias("avg_taxi_total_min")
    ) \
    .withColumn(
        "on_time_pct",
        F.round(F.col("on_time_flights") * 100.0 / F.col("total_flights"), 2)
    ) \
    .withColumn(
        "cancellation_rate",
        F.round(F.col("cancelled_flights") * 100.0 / F.col("total_flights"), 2)
    ) \
    .withColumn(
        "route_date_key",
        F.concat_ws("_", F.col("route_id"), F.col("date_key").cast("string"))
    )

fact_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("date_key") \
    .saveAsTable("Skylens_macro.team1.fact_daily_route_summary")

print("✅ fact_daily_route_summary done —", fact_daily.count(), "rows")

In [0]:
%sql
SELECT 'dim_date'                 AS table_name, COUNT(*) AS row_count FROM Skylens_macro.team1.dim_date
UNION ALL
SELECT 'dim_airline',              COUNT(*) FROM Skylens_macro.team1.dim_airline
UNION ALL
SELECT 'dim_origin_airport',       COUNT(*) FROM Skylens_macro.team1.dim_origin_airport
UNION ALL
SELECT 'dim_destination_airport',  COUNT(*) FROM Skylens_macro.team1.dim_destination_airport
UNION ALL
SELECT 'dim_aircraft',             COUNT(*) FROM Skylens_macro.team1.dim_aircraft
UNION ALL
SELECT 'dim_cancellation_reason',  COUNT(*) FROM Skylens_macro.team1.dim_cancellation_reason
UNION ALL
SELECT 'dim_delay_type',           COUNT(*) FROM Skylens_macro.team1.dim_delay_type
UNION ALL
SELECT 'fact_flights',             COUNT(*) FROM Skylens_macro.team1.fact_flights
UNION ALL
SELECT 'fact_daily_route_summary', COUNT(*) FROM Skylens_macro.team1.fact_daily_route_summary;

In [0]:
# check what origin airports exist in flights
df_flight_airports = df_silver_flights.select(
    F.col("ORIGIN_AIRPORT").alias("airport_code")
).union(
    df_silver_flights.select(
        F.col("DESTINATION_AIRPORT").alias("airport_code")
    )
).distinct()

# check what airports exist in your dim
df_dim_airports = spark.read.table("Skylens_macro.team1.dim_origin_airport") \
    .select("iata_code")

# find the ones that are in flights but NOT in dim
missing_airports = df_flight_airports \
    .join(df_dim_airports, df_flight_airports.airport_code == df_dim_airports.iata_code, "left_anti")

missing_airports.show(50, truncate=False)
print("Total missing airport codes:", missing_airports.count())

In [0]:
df_silver_airports.printSchema()
df_silver_airports.show(5, truncate=False)
df_bronze_airports = spark.read.table("Skylens_macro.team1.bronze_airports")
df_bronze_airports.printSchema()
df_bronze_airports.show(5, truncate=False)
df_raw_airports = spark.read.csv(
    "/Volumes/Skylens_macro/team1/preprocess_data/airports/",
    header=True,
    inferSchema=True
)

df_raw_airports.printSchema()
df_raw_airports.show(3, truncate=False)